# 🧠 Assignment: Support Vector Machine (SVM)
**Subject:** Machine Learning  
**Topic:** SVM with Linear and Polynomial Kernels  
**Dataset:** Breast Cancer Wisconsin (Diagnostic)

---

## 🎯 Objective
The goal of this assignment is to implement and compare **Support Vector Machine (SVM)** classifiers using **Linear** and **Polynomial** kernels. We aim to classify tumors as Malignant ($1$) or Benign ($0$) using only two features: `radius_mean` and `texture_mean`.

---

## 🛠️ Implementation Summary

### Task 1: Data Preparation
* **Dataset:** `data.csv` (UCI Breast Cancer Wisconsin).
* **Cleaning:** Removed `id` and `Unnamed: 32`.
* **Encoding:** Mapped diagnosis to binary values ($M = 1, B = 0$).
* **Feature Engineering:** Selected two primary features for 2D visualization.
* **Preprocessing:** Applied `StandardScaler` to ensure the SVM maximizes the margin effectively.

### Task 2: Model Training
We trained two distinct SVM models using `scikit-learn`:
1.  **Linear SVM:** Uses a linear hyperplane to separate classes.
2.  **Polynomial SVM:** Uses a 3rd-degree polynomial kernel to handle non-linear separation.

---

## 📊 Results and Visualizations

### Decision Boundaries
The plots generated below show how each model "draws the line" between the two classes. 

### Confusion Matrices
These matrices show the performance of the models in terms of **True Positives**, **True Negatives**, **False Positives**, and **False Negatives**.

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
# ── Task 1: Load and Prepare Data ────────────────────────────────────────────
# Load dataset
data = pd.read_csv('Dataset/data.csv')

In [ ]:
data.head()  # Display first few rows of the dataset

In [ ]:
# Clean data: drop unnecessary columns
data = data.drop(['id', 'Unnamed: 32'], axis=1)


In [ ]:
data.shape  # Check the shape of the dataset

In [ ]:

# Encode target: Malignant = 1, Benign = 0
data['diagnosis'] = data['diagnosis'].map({'M': 1, 'B': 0})


In [ ]:
data.sample(n=5)  # Display 5 random samples from the dataset

In [ ]:
# Select features and target
X = data[['radius_mean', 'texture_mean']].values
y = data['diagnosis'].values


In [ ]:
print(X[:5])  # Display first 5 rows of features
print(y[:5])  # Display first 5 target values

In [ ]:
# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
print(X_train[:5])  # Display first 5 rows of training features
print(y_train[:5])  # Display first 5 training target values

In [ ]:
# Feature Scaling (Crucial for SVM performance)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) # Fit scaler on training data and transform
X_test  = scaler.transform(X_test) # Use same scaler to transform test data , why not fit_transform on test data? because we want to use the same scaling parameters (mean and std) from training data to ensure consistency in feature scaling between training and testing datasets.


In [ ]:
print(X_train[:5])  # Display first 5 rows of scaled training features
print(X_test[:5])   # Display first 5 rows of scaled testing features

In [ ]:
# ── Task 2: Train SVM Models ──────────────────────────────────────────────────
# TODO: Train linear SVM
model_linear = SVC(kernel='linear') # Create SVM model with linear kernel
model_linear.fit(X_train, y_train) # Fit the linear SVM model to the training data

In [ ]:
# TODO: Train polynomial SVM (degree=3) 
model_poly = SVC(kernel='poly', degree=3) # Create SVM model with polynomial kernel of degree 3
model_poly.fit(X_train, y_train) # Fit the polynomial SVM model to the training data

In [ ]:
# Print accuracy results
print(f"Linear Kernel Accuracy : {accuracy_score(y_test, model_linear.predict(X_test)) * 100:.2f}%") # Calculate and print accuracy of linear SVM model on test data
print(f"Poly   Kernel Accuracy : {accuracy_score(y_test, model_poly.predict(X_test))   * 100:.2f}%") # Calculate and print accuracy of polynomial SVM model on test data


In [ ]:
# ── Task 3: Decision Boundary Plot ────────────────────────────────────────────
def plot_decision_boundary(model, X, y, title, ax):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[y == 0, 0], X[y == 0, 1], label='Benign',    color='blue',   edgecolors='k', s=40)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], label='Malignant', color='red',    edgecolors='k', s=40)
    ax.set_title(title)
    ax.set_xlabel('radius_mean (normalized)')
    ax.set_ylabel('texture_mean (normalized)')
    ax.legend()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_decision_boundary(model_linear, X_test, y_test, 'SVM — Linear Kernel',     axes[0])
plot_decision_boundary(model_poly,   X_test, y_test, 'SVM — Polynomial Kernel', axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# ── Task 4: Confusion Matrix Plot ─────────────────────────────────────────────
def plot_cm(model, X, y, title, ax):
    cm = confusion_matrix(y, model.predict(X))
    ax.imshow(cm, cmap='Blues')
    ax.set_title(title)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred B', 'Pred M'])
    ax.set_yticklabels(['True B', 'True M'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=14)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_cm(model_linear, X_test, y_test, 'Confusion Matrix — Linear', axes[0])
plot_cm(model_poly,   X_test, y_test, 'Confusion Matrix — Poly',   axes[1])
plt.tight_layout()
plt.show()

## 📝 Questions & Answers

#### **Q1. Which kernel gave higher accuracy — Linear or Polynomial? Why?**
**Answer:** Usually, the Linear kernel performs quite well or even slightly better on this specific subset of two features (radius and texture). This is because the breast cancer dataset is often linearly separable in lower dimensions. If the Polynomial kernel has lower accuracy, it might be due to overfitting or because the specific degree (3) creates a boundary that is too complex for the simplified 2-feature dataset.

#### **Q2. How is the boundary shape different between Linear and Polynomial?**
**Answer:** 
**Linear Kernel:** The decision boundary is a straight line. It tries to find the maximum margin hyperplane to separate the two classes.

**Polynomial Kernel:** The decision boundary is curved. Because it uses a higher-order transformation, it can "bend" the boundary to capture clusters of data that a straight line cannot reach.

#### **Q3. What does the confusion matrix tell you about False Positives and False Negatives? Which is more dangerous?**
**Answer:** 
**False Positive (FP):** Predicting Malignant (1) when it is actually Benign (0). This leads to unnecessary stress and further testing.

**False Negative (FN):** Predicting Benign (0) when it is actually Malignant (1). This is far more dangerous because a patient with cancer might go untreated, allowing the disease to progress. In medical diagnostics, minimizing False Negatives (increasing Recall) is usually the priority.

#### **Q4. What happens if you increase the degree to 5 or 10?**
**Answer:** 
When you increase the degree (e.g., degree=10), the boundary becomes extremely complex and "wiggly."

**Observation:** When I increase the degree to 10 I see the accuracy decrease on the test set from 82.46% to 771.93% .

**Why:** This is a classic case of Overfitting. The model tries so hard to capture every single point in the training data that it loses its ability to generalize to new, unseen data.

---
